# Self-Hosted Multi-Agent Memory [Optional]

When several agents collaborate — triage, policy lookup, order lookup — they need **shared memory** you control. **Self-hosted** means the memory layer runs on **your** infrastructure (Postgres, Redis, vector DB) instead of a vendor SaaS black box.

```
  Customer ticket thread
         │
         ▼
  ┌──────────────┐     read/write     ┌─────────────────────┐
  │ Triage Agent │◄──────────────────►│ Shared Memory Board │
  └──────┬───────┘                    │  • case facts       │
         │ handoff                    │  • agent notes      │
         ▼                            │  • policy citations │
  ┌──────────────┐                    └──────────▲──────────┘
  │ Policy Agent │──────────────────────────────┘
  └──────────────┘
         │
         ▼
  ┌──────────────┐
  │ Order Agent  │  private scratch per agent + shared board
  └──────────────┘
```

This optional notebook:

1. Defines **multi-agent memory scopes** (private, shared, durable)
2. Implements a **self-hosted memory service** in plain Python (JSON file backing)
3. Runs a **ShopCo Support Hub** with three agents on one shared board
4. Covers **handoffs, ACLs, and production mapping** (Postgres + Redis + vector store)

In [ ]:
"""
Self-hosted multi-agent memory lab — ShopCo Support Hub.
"""

import json
import os
import re
import tempfile
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any, Literal

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False

MEMORY_ROOT = Path(tempfile.gettempdir()) / "shopco_memory_lab"
MEMORY_ROOT.mkdir(parents=True, exist_ok=True)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


POLICY_SNIPPETS = {
    "returns": "Returns within 30 days with receipt. [pol-returns-01]",
    "shipping": "Free shipping on orders over $50. [pol-shipping-02]",
}

ORDERS_DB = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped", "customer": "Alice"},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing", "customer": "Bob"},
}

print(f"Memory root: {MEMORY_ROOT}")

---

## 1. Memory Scopes in Multi-Agent Systems

| Scope | Owner | Lifetime | Example |
|-------|-------|----------|---------|
| **Private scratch** | One agent | One turn / subgraph | Triage reasoning notes |
| **Thread checkpoint** | Orchestrator | One ticket thread | Messages, `order_id` |
| **Shared board** | All agents in workflow | Case duration | Facts agreed by specialists |
| **User profile store** | Platform | Cross-thread | Preferences, tier |
| **Archival / vector** | Platform | Months+ | Past cases, policy corpus |

Self-hosting matters when **shared board + checkpoints** must stay inside your VPC with audit logs.

---

## 2. Self-Hosted Stack (Conceptual)

| Component | Dev simulation | Production self-hosted |
|-----------|----------------|------------------------|
| Thread checkpoints | JSON files | **Postgres** (`PostgresSaver`) |
| Shared key-value board | JSON namespace files | **Postgres** / **Redis** |
| Pub/sub handoff events | In-process list | **Redis** streams / NATS |
| Semantic recall | Keyword search | **Qdrant** / **pgvector** |
| Secrets / ACL | Python dict | IAM + DB roles |

The code below uses **JSON files** so you can inspect persistence without standing up servers.

---

## 3. Namespace Model and ACLs

Each write targets `(namespace, key)`. Agents declare **read/write** permissions per namespace prefix.

In [ ]:
class Access(str, Enum):
    READ = "read"
    WRITE = "write"


@dataclass
class AgentMemoryProfile:
    agent_id: str
    read_prefixes: list[str]
    write_prefixes: list[str]

    def can_read(self, namespace: str) -> bool:
        return any(namespace.startswith(p) for p in self.read_prefixes)

    def can_write(self, namespace: str) -> bool:
        return any(namespace.startswith(p) for p in self.write_prefixes)


TRIAGE_PROFILE = AgentMemoryProfile(
    agent_id="triage",
    read_prefixes=["case/", "user/", "board/"],
    write_prefixes=["case/", "board/"],
)

POLICY_PROFILE = AgentMemoryProfile(
    agent_id="policy",
    read_prefixes=["case/", "board/", "policy/"],
    write_prefixes=["board/", "agent/policy/"],
)

ORDER_PROFILE = AgentMemoryProfile(
    agent_id="order",
    read_prefixes=["case/", "board/", "orders/"],
    write_prefixes=["board/", "agent/order/"],
)

print("Triage can write board:", TRIAGE_PROFILE.can_write("board/facts"))

---

## 4. SelfHostedMemoryService — File-Backed Store

In [ ]:
@dataclass
class MemoryRecord:
    namespace: str
    key: str
    value: Any
    updated_by: str
    updated_at: str = field(default_factory=utc_now)
    version: int = 1


class SelfHostedMemoryService:
    """Simulates Postgres/Redis KV — one JSON file per namespace."""

    def __init__(self, root: Path) -> None:
        self.root = root
        self.root.mkdir(parents=True, exist_ok=True)
        self._audit: list[dict[str, Any]] = []

    def _path(self, namespace: str) -> Path:
        safe = namespace.replace("/", "__")
        return self.root / f"{safe}.json"

    def _load_ns(self, namespace: str) -> dict[str, MemoryRecord]:
        p = self._path(namespace)
        if not p.exists():
            return {}
        raw = json.loads(p.read_text(encoding="utf-8"))
        return {k: MemoryRecord(**v) for k, v in raw.items()}

    def _save_ns(self, namespace: str, data: dict[str, MemoryRecord]) -> None:
        p = self._path(namespace)
        serial = {k: v.__dict__ for k, v in data.items()}
        p.write_text(json.dumps(serial, indent=2), encoding="utf-8")

    def put(
        self,
        namespace: str,
        key: str,
        value: Any,
        actor: AgentMemoryProfile,
    ) -> MemoryRecord:
        if not actor.can_write(namespace):
            raise PermissionError(f"{actor.agent_id} cannot write {namespace}")
        bucket = self._load_ns(namespace)
        prev = bucket.get(key)
        ver = (prev.version + 1) if prev else 1
        rec = MemoryRecord(namespace, key, value, actor.agent_id, utc_now(), ver)
        bucket[key] = rec
        self._save_ns(namespace, bucket)
        self._audit.append({"op": "put", "namespace": namespace, "key": key, "by": actor.agent_id})
        return rec

    def get(self, namespace: str, key: str, actor: AgentMemoryProfile) -> MemoryRecord | None:
        if not actor.can_read(namespace):
            raise PermissionError(f"{actor.agent_id} cannot read {namespace}")
        return self._load_ns(namespace).get(key)

    def list_keys(self, namespace: str, actor: AgentMemoryProfile) -> list[str]:
        if not actor.can_read(namespace):
            raise PermissionError(f"{actor.agent_id} cannot read {namespace}")
        return list(self._load_ns(namespace).keys())

    def audit_tail(self, n: int = 5) -> list[dict[str, Any]]:
        return self._audit[-n:]


memory = SelfHostedMemoryService(MEMORY_ROOT / "store")
memory.put("board", "status", {"phase": "new"}, TRIAGE_PROFILE)
print("Wrote board/status:", memory.get("board", "status", TRIAGE_PROFILE).value)

---

## 5. Event Log — Handoff Bus

Agents publish **handoff events** so the orchestrator and peers know what changed. In production this is a Redis stream or message queue.

In [ ]:
@dataclass
class HandoffEvent:
    event_id: str
    case_id: str
    from_agent: str
    to_agent: str
    intent: str
    payload: dict[str, Any]
    ts: str = field(default_factory=utc_now)


class HandoffBus:
    def __init__(self, log_path: Path) -> None:
        self.log_path = log_path
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        if not self.log_path.exists():
            self.log_path.write_text("[]", encoding="utf-8")

    def publish(self, event: HandoffEvent) -> None:
        events = json.loads(self.log_path.read_text(encoding="utf-8"))
        events.append(event.__dict__)
        self.log_path.write_text(json.dumps(events, indent=2), encoding="utf-8")

    def since(self, case_id: str) -> list[HandoffEvent]:
        events = json.loads(self.log_path.read_text(encoding="utf-8"))
        return [HandoffEvent(**e) for e in events if e["case_id"] == case_id]


bus = HandoffBus(MEMORY_ROOT / "handoffs.json")
bus.publish(HandoffEvent(
    event_id="evt-1", case_id="CASE-42", from_agent="triage",
    to_agent="policy", intent="needs_return_policy", payload={"topic": "returns"},
))
print("Events:", [e.intent for e in bus.since("CASE-42")])

---

## 6. Shared Memory Board

The **board** is the contract surface: structured facts every agent can rely on.

In [ ]:
@dataclass
class CaseBoard:
    case_id: str
    customer_message: str = ""
    customer_name: str | None = None
    order_id: str | None = None
    intent: str | None = None
    policy_citations: list[str] = field(default_factory=list)
    order_status: str | None = None
    resolution: str | None = None
    notes: list[str] = field(default_factory=list)

    def to_dict(self) -> dict[str, Any]:
        return self.__dict__.copy()

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> "CaseBoard":
        return cls(**{k: v for k, v in data.items() if k in cls.__dataclass_fields__})


def load_board(memory: SelfHostedMemoryService, case_id: str, actor: AgentMemoryProfile) -> CaseBoard:
    rec = memory.get("board", case_id, actor)
    if rec is None:
        return CaseBoard(case_id=case_id)
    return CaseBoard.from_dict(rec.value)


def save_board(memory: SelfHostedMemoryService, board: CaseBoard, actor: AgentMemoryProfile) -> None:
    memory.put("board", board.case_id, board.to_dict(), actor)


demo_board = CaseBoard(case_id="CASE-99", customer_message="Where is my order?")
save_board(memory, demo_board, TRIAGE_PROFILE)
print(load_board(memory, "CASE-99", ORDER_PROFILE).customer_message)

---

## 7. Specialist Agents — Private + Shared Writes

In [ ]:
def triage_agent(board: CaseBoard, msg: str) -> tuple[CaseBoard, HandoffEvent | None]:
    board.customer_message = msg
    name_m = re.search(r"my name is ([A-Za-z]+)", msg, re.I)
    if name_m:
        board.customer_name = name_m.group(1)
    oid = re.search(r"ORD-[0-9]{4}", msg.upper())
    if oid:
        board.order_id = oid.group(0)
    low = msg.lower()
    if "return" in low:
        board.intent = "return_question"
        return board, HandoffEvent(
            event_id=f"evt-{uuid.uuid4().hex[:6]}", case_id=board.case_id,
            from_agent="triage", to_agent="policy",
            intent="needs_return_policy", payload={"topic": "returns"},
        )
    if "order" in low or "ship" in low or board.order_id:
        board.intent = "order_status"
        return board, HandoffEvent(
            event_id=f"evt-{uuid.uuid4().hex[:6]}", case_id=board.case_id,
            from_agent="triage", to_agent="order",
            intent="needs_order_lookup", payload={"order_id": board.order_id},
        )
    board.intent = "general"
    board.resolution = "Thanks for contacting ShopCo. How can we help further?"
    return board, None


def policy_agent(board: CaseBoard) -> CaseBoard:
    topic = "returns" if board.intent == "return_question" else "shipping"
    cite = POLICY_SNIPPETS[topic]
    board.policy_citations.append(cite)
    board.notes.append(f"policy: cited {topic}")
    board.resolution = f"Per policy: {cite}"
    return board


def order_agent(board: CaseBoard) -> CaseBoard:
    oid = board.order_id or ""
    rec = ORDERS_DB.get(oid, {"status": "unknown"})
    board.order_status = rec["status"]
    board.notes.append(f"order: looked up {oid or 'none'}")
    board.resolution = f"Order {oid or 'N/A'} status: {board.order_status}."
    return board


b = CaseBoard(case_id="CASE-DEMO")
b, ev = triage_agent(b, "My name is Alice. Can I return ORD-1001?")
print("Handoff:", ev.to_agent if ev else None, "intent:", b.intent)

---

## 8. Multi-Agent Orchestrator with Shared Memory

In [ ]:
@dataclass
class OrchestrationResult:
    case_id: str
    board: CaseBoard
    agents_run: list[str]
    handoffs: list[HandoffEvent]
    audit: list[dict[str, Any]]


class ShopCoMemoryOrchestrator:
    def __init__(self, memory: SelfHostedMemoryService, bus: HandoffBus) -> None:
        self.memory = memory
        self.bus = bus

    def handle_ticket(self, case_id: str, message: str) -> OrchestrationResult:
        board = load_board(self.memory, case_id, TRIAGE_PROFILE)
        agents_run: list[str] = []
        handoffs: list[HandoffEvent] = []

        board, evt = triage_agent(board, message)
        save_board(self.memory, board, TRIAGE_PROFILE)
        agents_run.append("triage")

        if evt:
            self.bus.publish(evt)
            handoffs.append(evt)
            if evt.to_agent == "policy":
                board = policy_agent(board)
                save_board(self.memory, board, POLICY_PROFILE)
                agents_run.append("policy")
            elif evt.to_agent == "order":
                board = order_agent(board)
                save_board(self.memory, board, ORDER_PROFILE)
                agents_run.append("order")

        return OrchestrationResult(
            case_id=case_id,
            board=board,
            agents_run=agents_run,
            handoffs=handoffs,
            audit=self.memory.audit_tail(8),
        )


orch = ShopCoMemoryOrchestrator(memory, bus)
r_return = orch.handle_ticket("CASE-1001", "My name is Alice. Can I return ORD-1001?")
print("Agents:", r_return.agents_run)
print("Resolution:", r_return.board.resolution)

---

## 9. Demo — Return vs Order Paths

In [ ]:
r_order = orch.handle_ticket("CASE-1002", "Hi, status of ORD-1002 please.")
print("Order path agents:", r_order.agents_run)
print("Board notes:", r_order.board.notes)
print("Resolution:", r_order.board.resolution)

print("\nHandoff log for CASE-1002:")
for e in bus.since("CASE-1002"):
    print(f"  {e.from_agent} → {e.to_agent}: {e.intent}")

---

## 10. Thread Checkpoints vs Shared Board

| Concern | Thread checkpoint | Shared board |
|---------|-------------------|--------------|
| Key | `thread_id` | `case_id` |
| Consumers | One graph resume | Many agents |
| Content | Raw messages | Curated facts |
| Self-hosted | Postgres checkpointer table | KV / JSON column |

Use **checkpoints** for HITL resume; use the **board** for agent-to-agent truth.

In [ ]:
@dataclass
class ThreadCheckpoint:
    thread_id: str
    messages: list[dict[str, str]]
    step: int
    updated_at: str = field(default_factory=utc_now)


class ThreadCheckpointStore:
    def __init__(self, path: Path) -> None:
        self.path = path
        self.path.parent.mkdir(parents=True, exist_ok=True)
        if not self.path.exists():
            self.path.write_text("{}", encoding="utf-8")

    def save(self, cp: ThreadCheckpoint) -> None:
        data = json.loads(self.path.read_text(encoding="utf-8"))
        data[cp.thread_id] = cp.__dict__
        self.path.write_text(json.dumps(data, indent=2), encoding="utf-8")

    def load(self, thread_id: str) -> ThreadCheckpoint | None:
        data = json.loads(self.path.read_text(encoding="utf-8"))
        raw = data.get(thread_id)
        return ThreadCheckpoint(**raw) if raw else None


ckpt_store = ThreadCheckpointStore(MEMORY_ROOT / "checkpoints.json")
ckpt_store.save(ThreadCheckpoint(
    thread_id="thread-CASE-1001",
    messages=[{"role": "user", "content": "Can I return ORD-1001?"}],
    step=1,
))
board_snapshot = load_board(memory, "CASE-1001", TRIAGE_PROFILE)
print("Checkpoint thread:", ckpt_store.load("thread-CASE-1001").step)
print("Board resolution:", board_snapshot.resolution)

---

## 11. Conflict Resolution — Last-Writer vs Merge

When two agents update the board, pick a strategy:

- **Last-writer-wins** — simple, may lose data
- **Field-level merge** — each agent owns fields (`policy_citations` vs `order_status`)
- **Optimistic locking** — reject writes if `version` changed (see `MemoryRecord.version`)

In [ ]:
def merge_board(existing: CaseBoard, patch: dict[str, Any]) -> CaseBoard:
    data = existing.to_dict()
    for k, v in patch.items():
        if k == "notes" and isinstance(v, list):
            data["notes"] = data.get("notes", []) + v
        elif k == "policy_citations" and isinstance(v, list):
            data["policy_citations"] = data.get("policy_citations", []) + v
        else:
            data[k] = v
    return CaseBoard.from_dict(data)


base = CaseBoard(case_id="CASE-MERGE", order_id="ORD-1001")
base = merge_board(base, {"order_status": "shipped", "notes": ["order: ok"]})
base = merge_board(base, {"policy_citations": [POLICY_SNIPPETS["returns"]], "notes": ["policy: ok"]})
print("Merged notes:", base.notes)
print("Citations + status:", base.policy_citations, base.order_status)

---

## 12. Semantic Recall Layer (Lightweight)

Archival search for past cases — production uses pgvector/Qdrant; here keyword match simulates the API.

In [ ]:
class ArchivalCaseIndex:
    def __init__(self) -> None:
        self._docs: list[dict[str, str]] = []

    def index_case(self, board: CaseBoard) -> None:
        text = " ".join(filter(None, [
            board.customer_message, board.order_id or "",
            " ".join(board.policy_citations), board.resolution or "",
        ]))
        self._docs.append({"case_id": board.case_id, "text": text})

    def search(self, query: str, top_k: int = 2) -> list[dict[str, str]]:
        terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
        scored: list[tuple[int, dict[str, str]]] = []
        for d in self._docs:
            score = sum(1 for t in terms if t in d["text"].lower()) if terms else 0
            if score:
                scored.append((score, d))
        scored.sort(key=lambda x: -x[0])
        return [d for _, d in scored[:top_k]]


index = ArchivalCaseIndex()
index.index_case(r_return.board)
index.index_case(r_order.board)
print("Similar cases for 'return Alice':", index.search("return Alice"))

---

## 13. Observability — Audit Trail

Self-hosted memory needs **who wrote what, when** for compliance.

In [ ]:
print("Recent memory audit:")
for row in memory.audit_tail(10):
    print(f"  {row['op']} {row['namespace']}/{row['key']} by {row['by']}")

---

## 14. Cloud SaaS vs Self-Hosted

| Dimension | Managed agent memory SaaS | Self-hosted |
|-----------|---------------------------|-------------|
| Time to ship | Fast | Slower setup |
| Data residency | Vendor cloud | Your VPC |
| Multi-agent board | Varies | You design namespaces |
| Cost at scale | Per-seat / per-token | Infra + ops |
| Custom ACLs | Limited | Full control |

Regulated industries (finance, health) often **require** self-hosted checkpoints and boards.

---

## 15. Optional Live LLM Triage

Set `USE_LIVE_LLM = True` to classify intent with `gpt-4o-mini` instead of rules.

In [ ]:
def classify_intent_live(message: str) -> Literal["return_question", "order_status", "general"]:
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    resp = llm.invoke([
        SystemMessage(content="Reply with one label: return_question, order_status, or general."),
        HumanMessage(content=message),
    ])
    label = str(resp.content).strip().lower()
    if label in ("return_question", "order_status", "general"):
        return label  # type: ignore[return-value]
    return "general"


msg = "Alice here — is ORD-1001 still eligible for return?"
if USE_LIVE_LLM:
    print("Live intent:", classify_intent_live(msg))
else:
    _, ev = triage_agent(CaseBoard(case_id="X"), msg)
  print("Rule intent handoff:", ev.to_agent if ev else "none")

---

## 16. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| One giant shared prompt | Agents overwrite context | Structured **board** fields |
| No ACLs | Policy agent edits orders | Namespace write profiles |
| Board = raw chat log | Retrieval noise | Curate facts on handoff |
| No event bus | Lost orchestration trace | Handoff log / Redis stream |
| In-memory only in prod | Restart loses cases | Postgres + durable bus |
| Last-writer on list fields | Dropped notes | Field-level **merge** |

---

## 17. Quiz

1. What is the difference between a thread checkpoint and a shared memory board?
2. Why do agents need `AgentMemoryProfile` ACLs?
3. What production store backs the JSON simulation in section 4?
4. When should you use a handoff bus?
5. Name two conflict strategies for concurrent board updates.

---

## 18. Summary

**Self-hosted multi-agent memory** combines **durable checkpoints** (per thread), a **shared board** (per case), **handoff events**, and optional **archival search** — all under your infrastructure and ACL policies.

This notebook ran ShopCo triage → policy/order specialists on a file-backed `SelfHostedMemoryService`, with audit logs and merge-friendly board updates.

In production, swap JSON files for **Postgres** (checkpoints + board), **Redis** (handoffs), and **pgvector/Qdrant** (case recall) while keeping the same namespace and ACL model.